# 05: Generator fingerprints made visible (Step 7)

No training. Two views of the traces each source leaves in the pixels, computed on the processed test set (500 images per generator, 3,000 real):

1. **Average frequency spectrum of the noise residual** per source (residual = image minus a slightly blurred copy).
2. **NPR residual** (Tan et al., CVPR 2024): image minus a copy that was down-sampled by 2 and up-sampled again (nearest neighbour).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

ROOT = Path("/content/drive/MyDrive/minor-project")
LOCAL = Path("/content/processed")
LOCAL.mkdir(exist_ok=True)
for name in ["test_images.npy", "test_meta.csv"]:
    if not (LOCAL / name).exists():
        shutil.copy(ROOT / "processed" / name, LOCAL / name)

images = np.load(LOCAL / "test_images.npy", mmap_mode="r")
meta = pd.read_csv(LOCAL / "test_meta.csv")
SOURCES = ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"]
print(meta["generator"].value_counts())

## 1. Average spectrum of the noise residual

In [ ]:
def to_gray(batch):
    return batch.astype(np.float32) @ np.array([0.299, 0.587, 0.114], dtype=np.float32)


def noise_residual(gray):
    return gray - gaussian_filter(gray, sigma=(0, 1, 1))


def mean_spectrum(rows, batch_size=100):
    total = np.zeros((224, 224), dtype=np.float64)
    for start in range(0, len(rows), batch_size):
        batch = images[np.sort(rows[start:start + batch_size])]
        residual = noise_residual(to_gray(batch))
        spectrum = np.abs(np.fft.fftshift(np.fft.fft2(residual), axes=(1, 2)))
        total += spectrum.sum(axis=0)
    return total / len(rows)


N_PER_SOURCE = 500
spectra = {}
for source in SOURCES:
    rows = meta.index[meta["generator"] == source].to_numpy()[:N_PER_SOURCE]
    spectra[source] = np.log(mean_spectrum(rows) + 1e-6)
    print(f"{source}: averaged {len(rows)} images")

In [ ]:
fig, axes = plt.subplots(2, len(SOURCES), figsize=(2.6 * len(SOURCES), 5.8))
low, high = np.percentile(np.stack(list(spectra.values())), [1, 99.5])
for col, source in enumerate(SOURCES):
    axes[0, col].imshow(spectra[source], cmap="inferno", vmin=low, vmax=high)
    axes[0, col].set_title(source)
    diff = spectra[source] - spectra["Real"]
    axes[1, col].imshow(diff, cmap="coolwarm", vmin=-1, vmax=1)
    for ax in axes[:, col]:
        ax.set_xticks([])
        ax.set_yticks([])
axes[0, 0].set_ylabel("log spectrum")
axes[1, 0].set_ylabel("minus Real")
plt.tight_layout()
plt.savefig(ROOT / "results" / "05_spectra.png", dpi=120)
plt.show()

## 2. NPR residual

In [ ]:
def npr(batch):
    x = batch.astype(np.float32)
    rebuilt = x[:, ::2, ::2].repeat(2, axis=1).repeat(2, axis=2)
    return x - rebuilt


examples = [meta.index[meta["generator"] == source][0] for source in SOURCES]
residuals = npr(images[examples])
ZOOM = slice(80, 144)

fig, axes = plt.subplots(2, len(SOURCES), figsize=(2.6 * len(SOURCES), 5.6))
for col, (source, idx) in enumerate(zip(SOURCES, examples)):
    axes[0, col].imshow(images[idx][ZOOM, ZOOM])
    axes[0, col].set_title(source)
    magnitude = np.abs(residuals[col][ZOOM, ZOOM]).mean(axis=2)
    axes[1, col].imshow(magnitude, cmap="gray", vmin=0, vmax=np.percentile(magnitude, 99))
    for ax in axes[:, col]:
        ax.set_xticks([])
        ax.set_yticks([])
axes[0, 0].set_ylabel("image (64x64 zoom)")
axes[1, 0].set_ylabel("|NPR|")
plt.tight_layout()
plt.savefig(ROOT / "results" / "05_npr_examples.png", dpi=120)
plt.show()

In [ ]:
energy = {}
for source in SOURCES:
    rows = np.sort(meta.index[meta["generator"] == source].to_numpy()[:N_PER_SOURCE])
    energy[source] = float(np.abs(npr(images[rows])).mean())

energy = pd.Series(energy, name="mean |NPR|").round(3)
display(energy.to_frame())
energy.to_csv(ROOT / "results" / "05_npr_energy.csv")